In [14]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, ndcg_score
from imblearn.over_sampling import SMOTE
import optuna
from optuna.samplers import TPESampler

RANDOM_STATE = 42

# Φόρτωση των συνόλων από το 03_models
train_fit = pd.read_pickle("train_fit.pkl")
valid     = pd.read_pickle("valid.pkl")
train     = pd.read_pickle("train.pkl")
test      = pd.read_pickle("test.pkl")
print("train_fit:", train_fit.shape, "| valid:", valid.shape, "| train:", train.shape, "| test:", test.shape)

# Μεταβλητές εισόδου που επιλέχθηκαν στο 03: raw_scaled (25 raw / ενεργητικό + at, prcc_f, csho)
raw_cols = ['act','ap','at','ceq','che','cogs','csho','dlc','dltis','dltt','dp','ib',
            'invt','ivao','ivst','lct','lt','ni','ppegt','pstk','re','rect','sale',
            'sstk','txp','txt','xint','prcc_f']
keep_as_is = ['at', 'prcc_f', 'csho']
scaled_cols = []
for c in raw_cols:
    if c not in keep_as_is:
        scaled_cols.append(c + '_at')
raw_scaled = scaled_cols + keep_as_is
print("Πλήθος raw_scaled:", len(raw_scaled))   # πρέπει 28

# NDCG@1% : ανά έτος, k = 1% των παρατηρήσεων του έτους, μέσος όρος των ετών
def ndcg_1pct(data, prediction):
    scores = []
    for year in sorted(data['fyear'].unique()):
        year_mask = (data['fyear'] == year).values
        k = round(0.01 * year_mask.sum())
        score = ndcg_score([data.loc[year_mask, 'misstate']], [prediction[year_mask]], k=k)
        scores.append(score)
    return np.mean(scores)

train_fit: (54052, 78) | valid: (13114, 78) | train: (67166, 78) | test: (35166, 78)
Πλήθος raw_scaled: 28


### **ΑΝΙΣΟΡΡΟΠΙΑ ΚΛΑΣΕΩΝ**

In [3]:
# Βάση: raw_scaled χωρίς χειρισμό ανισορροπίας 
model_base = XGBClassifier(random_state=RANDOM_STATE)
model_base.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_base = model_base.predict_proba(valid[raw_scaled])[:, 1]
auc_base = roc_auc_score(valid['misstate'], prediction_base)
ndcg_base = ndcg_1pct(valid, prediction_base)

# Παραλλαγή 1: scale_pos_weight (βάρος στις απάτες = μη απάτες / απάτες του train_fit)
n_neg = (train_fit['misstate'] == 0).sum()
n_pos = (train_fit['misstate'] == 1).sum()
spw = n_neg / n_pos
print("Μη απάτες:", n_neg, "| Απάτες:", n_pos, "| scale_pos_weight:", round(spw, 1))

model_spw = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=spw)
model_spw.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_spw = model_spw.predict_proba(valid[raw_scaled])[:, 1]
auc_spw = roc_auc_score(valid['misstate'], prediction_spw)
ndcg_spw = ndcg_1pct(valid, prediction_spw)

print("Καμία (βάση):      AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print("scale_pos_weight:  AUC", round(auc_spw, 4),  "| NDCG@1%", round(ndcg_spw, 4))

Μη απάτες: 53808 | Απάτες: 244 | scale_pos_weight: 220.5
Καμία (βάση):      AUC 0.7329 | NDCG@1% 0.1396
scale_pos_weight:  AUC 0.6973 | NDCG@1% 0.0617


##### Παρατηρούμε οτι για scale_pos_weight = 220.5 μειώθηκε και το AUC και το NDCG@1%

In [4]:
# scale_pos_weight: δοκιμή τιμών
spw_values = [5, 10, 25, 50, 100, round(spw, 1)]
spw_results = []

for w in spw_values:
    model_w = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=w)
    model_w.fit(train_fit[raw_scaled], train_fit['misstate'])
    prediction_w = model_w.predict_proba(valid[raw_scaled])[:, 1]
    spw_results.append((w,
                        roc_auc_score(valid['misstate'], prediction_w),
                        ndcg_1pct(valid, prediction_w)))

spw_table = pd.DataFrame(spw_results, columns=['scale_pos_weight', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(spw_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
   scale_pos_weight     AUC  NDCG@1%
0               5.0  0.7509   0.1135
1              10.0  0.7288   0.1142
2              25.0  0.7053   0.1033
3              50.0  0.7231   0.0664
4             100.0  0.7269   0.0791
5             220.5  0.6973   0.0617


##### Παρατηρούμε ότι για scale_pos_weight = 5 βελτιώνεται το AUC ενώ μειώνεται λίγο το NDCG@1% ενώ για όλα σε όλες τις υπόλοιπες δοκιμές μειώνονται και τα δύο σε σχέση με το αρχικό. Συνεπώς κρατλαμε το scale_pos_weight = 5 καθώς έχουμε ως βασικό μετρο το AUC και δευτεύον το NDCG@1%.

In [5]:
#  Random Undersampling (μόνο στο train_fit): κρατάμε όλες τις απάτες
# και τυχαίο δείγμα μη απατών, ώστε απάτες : μη απάτες = 1:1, 1:4, 1:10
frauds = train_fit[train_fit['misstate'] == 1]
non_frauds = train_fit[train_fit['misstate'] == 0]

rus_ratios = [1, 4, 10]          # πόσες μη απάτες για κάθε 1 απάτη
rus_results = []

for k in rus_ratios:
    non_sample = non_frauds.sample(n=len(frauds) * k, random_state=RANDOM_STATE)
    train_rus = pd.concat([frauds, non_sample])

    model_r = XGBClassifier(random_state=RANDOM_STATE)
    model_r.fit(train_rus[raw_scaled], train_rus['misstate'])
    prediction_r = model_r.predict_proba(valid[raw_scaled])[:, 1]

    rus_results.append(('1:' + str(k), len(train_rus),
                        roc_auc_score(valid['misstate'], prediction_r),
                        ndcg_1pct(valid, prediction_r)))

rus_table = pd.DataFrame(rus_results, columns=['Αναλογία', 'Γραμμές train', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(rus_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
  Αναλογία  Γραμμές train     AUC  NDCG@1%
0      1:1            488  0.7177   0.1331
1      1:4           1220  0.7386   0.1078
2     1:10           2684  0.7177   0.1162


 Παρατηρούμε ότι για random undersampling με αναλογια 1:4 παρουσίασε μεγαλύτερο AUC κατά 0,0057 ενώ μειώθηκε το NDCG@1%.Αυτό σημαίνει πως βελτιώθηκε η συνολική κατάταξη αλλα΄όχι η κορυφή της λίστας. Ωστόσο, το AUC 0.7386 αυτης της μεθόδου ήταν μικρότερο από αυτο του scale_pos_weight = 5 που ήταν ίσο με 0,7509

In [9]:
# Έλεγχος σταθερότητας: σκέτο RUS 1:4 με 5 διαφορετικές τυχαίες επιλογές
seeds = [1, 7, 21, 42, 100]
rus_seed_results = []

for s in seeds:
    non_sample = non_frauds.sample(n=len(frauds) * 4, random_state=s)
    train_r = pd.concat([frauds, non_sample])

    model_r = XGBClassifier(random_state=RANDOM_STATE)
    model_r.fit(train_r[raw_scaled], train_r['misstate'])
    prediction_r = model_r.predict_proba(valid[raw_scaled])[:, 1]
    rus_seed_results.append((s, roc_auc_score(valid['misstate'], prediction_r),
                             ndcg_1pct(valid, prediction_r)))

rus_seed_table = pd.DataFrame(rus_seed_results, columns=['random_state', 'AUC', 'NDCG@1%'])
print(rus_seed_table.round(4))
print("Μέσος όρος: AUC", round(rus_seed_table['AUC'].mean(), 4), "| NDCG@1%", round(rus_seed_table['NDCG@1%'].mean(), 4))
print("Ελάχιστο - μέγιστο AUC:", round(rus_seed_table['AUC'].min(), 4), "-", round(rus_seed_table['AUC'].max(), 4))

   random_state     AUC  NDCG@1%
0             1  0.7461   0.1137
1             7  0.7565   0.0773
2            21  0.7431   0.0963
3            42  0.7386   0.1078
4           100  0.7391   0.0993
Μέσος όρος: AUC 0.7447 | NDCG@1% 0.0989
Ελάχιστο - μέγιστο AUC: 0.7386 - 0.7565


Παρατηρούμε ότι γενικά το AUC έχει μια σταθερότητα με μέσο όρο 0,7447 ενώ το NDCG παρουσιάζει ένα λίγο μεγαλύτερο έυρος με μέσο όρο 0,0989

In [ ]:
# Το SMOTE δεν δέχεται κενές τιμές: συμπλήρωση με τη διάμεσο του train_fit.
# Οι ίδιες διάμεσοι μπαίνουν και στο validation (υπολογίζονται μόνο από το train_fit -> καμία διαρροή).
medians = train_fit[raw_scaled].median()
X_fit = train_fit[raw_scaled].fillna(medians)
y_fit = train_fit['misstate']
X_valid_filled = valid[raw_scaled].fillna(medians)

smote_results = []

# Έλεγχος: μόνο συμπλήρωση, χωρίς SMOTE (για να ξεχωρίσουμε την επίδραση της συμπλήρωσης)
model_imp = XGBClassifier(random_state=RANDOM_STATE)
model_imp.fit(X_fit, y_fit)
prediction_imp = model_imp.predict_proba(X_valid_filled)[:, 1]
smote_results.append(('Μόνο συμπλήρωση', int(y_fit.sum()), len(y_fit),
                      roc_auc_score(valid['misstate'], prediction_imp),
                      ndcg_1pct(valid, prediction_imp)))

# SMOTE (k_neighbors = 5, όπως Chawla et al., 2002) - μόνο στο train_fit
smote_ratios = [10, 4, 1]        # μη απάτες για κάθε 1 απάτη μετά το SMOTE
for k in smote_ratios:
    smote = SMOTE(sampling_strategy=1 / k, random_state=RANDOM_STATE)
    X_sm, y_sm = smote.fit_resample(X_fit, y_fit)

    model_s = XGBClassifier(random_state=RANDOM_STATE)
    model_s.fit(X_sm, y_sm)
    prediction_s = model_s.predict_proba(X_valid_filled)[:, 1]

    smote_results.append(('SMOTE 1:' + str(k), int(y_sm.sum()), len(y_sm),
                          roc_auc_score(valid['misstate'], prediction_s),
                          ndcg_1pct(valid, prediction_s)))

smote_table = pd.DataFrame(smote_results,
                           columns=['Παραλλαγή', 'Απάτες στο train', 'Γραμμές train', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(smote_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
         Παραλλαγή  Απάτες στο train  Γραμμές train     AUC  NDCG@1%
0  Μόνο συμπλήρωση               244          54052  0.7322   0.1099
1       SMOTE 1:10              5380          59188  0.6846   0.0379
2        SMOTE 1:4             13452          67260  0.6810   0.0349
3        SMOTE 1:1             53808         107616  0.6562   0.0380


 Η μέθοδος SMOTE αρχικά δεν μπορεί να λειρουργήσει με κενές τιμές οπότε συμπληρώθηκε η διάμεσος για τις ελλειπούσες τιμες με αποτέλεσμα το AUC να μειωθεί από 0,7329 σε 0,7322. Επίσης, δεν βελτίωσε με καμία αναλογία το αρχικό μοντέλο, αντίθετα μειώθηκε αρκετα.

In [7]:
# Β3 - Συνδυασμός: Random Undersampling και μετά SMOTE (Chawla et al., 2002) - ΜΟΝΟ στο train_fit
combos = [(10, 4), (10, 1), (4, 1)]   # (μη απάτες ανά απάτη μετά το undersampling, μετά το SMOTE)
combo_results = []

for k_rus, k_smote in combos:
    # 1) Undersampling: όλες οι απάτες + τυχαίο δείγμα μη απατών
    non_sample = non_frauds.sample(n=len(frauds) * k_rus, random_state=RANDOM_STATE)
    train_c = pd.concat([frauds, non_sample])

    # 2) Συμπλήρωση κενών (με τις ίδιες διαμέσους του train_fit) και SMOTE
    X_c = train_c[raw_scaled].fillna(medians)
    y_c = train_c['misstate']
    X_cs, y_cs = SMOTE(sampling_strategy=1 / k_smote, random_state=RANDOM_STATE).fit_resample(X_c, y_c)

    # 3) Εκπαίδευση και αξιολόγηση στο κανονικό validation
    model_c = XGBClassifier(random_state=RANDOM_STATE)
    model_c.fit(X_cs, y_cs)
    prediction_c = model_c.predict_proba(X_valid_filled)[:, 1]

    combo_results.append(('RUS 1:' + str(k_rus) + ' → SMOTE 1:' + str(k_smote),
                          int(y_cs.sum()), len(y_cs),
                          roc_auc_score(valid['misstate'], prediction_c),
                          ndcg_1pct(valid, prediction_c)))

combo_table = pd.DataFrame(combo_results,
                           columns=['Συνδυασμός', 'Απάτες στο train', 'Γραμμές train', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(combo_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
             Συνδυασμός  Απάτες στο train  Γραμμές train     AUC  NDCG@1%
0  RUS 1:10 → SMOTE 1:4               610           3050  0.7184   0.0971
1  RUS 1:10 → SMOTE 1:1              2440           4880  0.6993   0.0591
2   RUS 1:4 → SMOTE 1:1               976           1952  0.7474   0.1500


In [8]:
# Έλεγχος σταθερότητας: RUS 1:4 → SMOTE 1:1 με 5 διαφορετικές τυχαίες επιλογές
seeds = [1, 7, 21, 42, 100]
seed_results = []

for s in seeds:
    non_sample = non_frauds.sample(n=len(frauds) * 4, random_state=s)
    train_c = pd.concat([frauds, non_sample])
    X_c = train_c[raw_scaled].fillna(medians)
    y_c = train_c['misstate']
    X_cs, y_cs = SMOTE(sampling_strategy=1.0, random_state=s).fit_resample(X_c, y_c)

    model_c = XGBClassifier(random_state=RANDOM_STATE)
    model_c.fit(X_cs, y_cs)
    prediction_c = model_c.predict_proba(X_valid_filled)[:, 1]
    seed_results.append((s, roc_auc_score(valid['misstate'], prediction_c),
                         ndcg_1pct(valid, prediction_c)))

seed_table = pd.DataFrame(seed_results, columns=['random_state', 'AUC', 'NDCG@1%'])
print(seed_table.round(4))
print("Μέσος όρος: AUC", round(seed_table['AUC'].mean(), 4), "| NDCG@1%", round(seed_table['NDCG@1%'].mean(), 4))
print("Ελάχιστο - μέγιστο AUC:", round(seed_table['AUC'].min(), 4), "-", round(seed_table['AUC'].max(), 4))

   random_state     AUC  NDCG@1%
0             1  0.7469   0.0683
1             7  0.7630   0.1035
2            21  0.7315   0.0902
3            42  0.7474   0.1500
4           100  0.7554   0.0575
Μέσος όρος: AUC 0.7488 | NDCG@1% 0.0939
Ελάχιστο - μέγιστο AUC: 0.7315 - 0.763


Παρατηρούμε ότι ο μέσος όρος του AUC είναι σχεδόν ίδιος με τον αρχικό και γενικά έχει σχετικά μικρό εύρος (0,7315 - 0,7630) σε αντίθεση με το NDCG@1% το οποίο είναι αρκετά ασταθές και ήταν τυχαίο πριν το 0,15. Ο μέσος όρος είναι 0,0939 που βρίσκεται κάτω από το βασικό μοντέλο (model_4_scaled_raw με NDCG@1% 0,1396 ) και κάτω από το scale_pos_weight = 5 με NDCG@1% = 0,1135. Άρα το αποτέλεσμα του συνδυασμού στο NDCG δεν ήταν πραγματικό αλλά τυχαίο. 


In [10]:
# Αντίστροφη σειρά: πρώτα SMOTE (σε όλο το train_fit), μετά Random Undersampling
reverse_combos = [(10, 4), (10, 1), (4, 1)]   # (αναλογία μετά το SMOTE, αναλογία μετά το undersampling)
reverse_results = []

for k_smote, k_rus in reverse_combos:
    # 1) SMOTE στο πλήρες train_fit (με τα ήδη συμπληρωμένα X_fit, y_fit)
    X_s, y_s = SMOTE(sampling_strategy=1 / k_smote, random_state=RANDOM_STATE).fit_resample(X_fit, y_fit)

    # Ενώνουμε X και y σε ένα πίνακα για να μπορούμε να χωρίσουμε απάτες / μη απάτες
    data_s = X_s.copy()
    data_s['misstate'] = y_s.values
    frauds_s = data_s[data_s['misstate'] == 1]
    non_frauds_s = data_s[data_s['misstate'] == 0]

    # 2) Undersampling: όλες οι απάτες (πραγματικές + τεχνητές) + τυχαίο δείγμα μη απατών
    non_sample_s = non_frauds_s.sample(n=len(frauds_s) * k_rus, random_state=RANDOM_STATE)
    train_sr = pd.concat([frauds_s, non_sample_s])

    # 3) Εκπαίδευση και αξιολόγηση στο κανονικό validation
    model_sr = XGBClassifier(random_state=RANDOM_STATE)
    model_sr.fit(train_sr[raw_scaled], train_sr['misstate'])
    prediction_sr = model_sr.predict_proba(X_valid_filled)[:, 1]

    reverse_results.append(('SMOTE 1:' + str(k_smote) + ' → RUS 1:' + str(k_rus),
                            int(train_sr['misstate'].sum()), len(train_sr),
                            roc_auc_score(valid['misstate'], prediction_sr),
                            ndcg_1pct(valid, prediction_sr)))

reverse_table = pd.DataFrame(reverse_results,
                             columns=['Συνδυασμός', 'Απάτες στο train', 'Γραμμές train', 'AUC', 'NDCG@1%'])
print("Βάση (καμία): AUC", round(auc_base, 4), "| NDCG@1%", round(ndcg_base, 4))
print(reverse_table.round(4))

Βάση (καμία): AUC 0.7329 | NDCG@1% 0.1396
             Συνδυασμός  Απάτες στο train  Γραμμές train     AUC  NDCG@1%
0  SMOTE 1:10 → RUS 1:4              5380          26900  0.6838   0.0425
1  SMOTE 1:10 → RUS 1:1              5380          10760  0.7020   0.0403
2   SMOTE 1:4 → RUS 1:1             13452          26904  0.6873   0.0429


Το AUC με τον συνδυασμό SMOTE και μετά undersampling είναι αρκετά μικρότερο των προηγούμενων παραλλαγών όπως και το NDCG@1%. 

Συνεπώς το **scale_pos_weight = 5** είναι *νικητής* με AUC  0.7509 και NDCG@1% = 0.1135. Σημειώνεται επίσης πως δεν εχει τυχαιότητα το μοντέλο αυτο και χρησιμοποιεί όλα τα δεδομένα.
H δεύτερη καλύτερη επιλογή είναι ο συνδυασμός RUS 1:4 → SMOTE 1:1 με AUC 0.7488 και NDCG@1% = 0,0939

## **OPTUNA**

### **Εκδοχή Α**: scale_pos_weight σταθερό = 5 (νικητής), το Optuna ψάχνει τις 6 ρυθμίσεις

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)   # να μην τυπώνει γραμμή για κάθε δοκιμή

def objective_spw_fixed(trial):
    n_estimators     = trial.suggest_int('n_estimators', 100, 2000)
    learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth        = trial.suggest_int('max_depth', 3, 12)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 7)
    subsample        = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)

    model = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=5,
                          n_estimators=n_estimators, learning_rate=learning_rate,
                          max_depth=max_depth, min_child_weight=min_child_weight,
                          subsample=subsample, colsample_bytree=colsample_bytree)
    model.fit(train_fit[raw_scaled], train_fit['misstate'])
    prediction = model.predict_proba(valid[raw_scaled])[:, 1]
    return roc_auc_score(valid['misstate'], prediction)

study_A_spw = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_A_spw.optimize(objective_spw_fixed, n_trials=50)

print("Καλύτερο AUC στο validation:", round(study_A_spw.best_value, 4))
print("Καλύτερες ρυθμίσεις:")
print(study_A_spw.best_params)

Καλύτερο AUC στο validation: 0.7905
Καλύτερες ρυθμίσεις:
{'n_estimators': 496, 'learning_rate': 0.015258454459097485, 'max_depth': 4, 'min_child_weight': 2.599630437518745, 'subsample': 0.5219865275854596, 'colsample_bytree': 0.4395595092783027}


In [16]:
# Εκδοχή Α: ξαναεκπαίδευση με τις καλύτερες ρυθμίσεις -> έλεγχος AUC και υπολογισμός NDCG@1%
best_A_spw = study_A_spw.best_params

model_A_spw = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=5,
                            n_estimators=best_A_spw['n_estimators'],
                            learning_rate=best_A_spw['learning_rate'],
                            max_depth=best_A_spw['max_depth'],
                            min_child_weight=best_A_spw['min_child_weight'],
                            subsample=best_A_spw['subsample'],
                            colsample_bytree=best_A_spw['colsample_bytree'])
model_A_spw.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_A_spw = model_A_spw.predict_proba(valid[raw_scaled])[:, 1]

auc_A_spw = roc_auc_score(valid['misstate'], prediction_A_spw)
ndcg_A_spw = ndcg_1pct(valid, prediction_A_spw)
print("scale_pos_weight=5, προεπιλογές:  AUC 0.7509 | NDCG@1% 0.1135")
print("scale_pos_weight=5, Optuna (Α):   AUC", round(auc_A_spw, 4), "| NDCG@1%", round(ndcg_A_spw, 4))

scale_pos_weight=5, προεπιλογές:  AUC 0.7509 | NDCG@1% 0.1135
scale_pos_weight=5, Optuna (Α):   AUC 0.7905 | NDCG@1% 0.1579


Με τη ρύθμιση των υπερπαραμέτρων (Optuna, 50 δοκιμές), το AUC αυξήθηκε από 0,7509 σε 0,7905 και το NDCG@1% από 0,1135 σε 0,1579. Οι ρυθμίσεις που επιλέχθηκαν (μικρό learning_rate, ρηχά δέντρα, χρήση μέρους των γραμμών και των μεταβλητών σε κάθε δέντρο) οδηγούν σε απλούστερο μοντέλο, κάτι που ταιριάζει με τον μικρό αριθμό απατών του δείγματος.

### **Εκδοχή Β** :και το scale_pos_weight μέσα στο Optuna

In [17]:
# Εκδοχή Β: το Optuna ψάχνει και το scale_pos_weight μαζί με τις 6 ρυθμίσεις
def objective_spw_tuned(trial):
    scale_pos_weight = trial.suggest_float('scale_pos_weight', 1, 220.5, log=True)
    n_estimators     = trial.suggest_int('n_estimators', 100, 2000)
    learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth        = trial.suggest_int('max_depth', 3, 12)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 7)
    subsample        = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)

    model = XGBClassifier(random_state=RANDOM_STATE, scale_pos_weight=scale_pos_weight,
                          n_estimators=n_estimators, learning_rate=learning_rate,
                          max_depth=max_depth, min_child_weight=min_child_weight,
                          subsample=subsample, colsample_bytree=colsample_bytree)
    model.fit(train_fit[raw_scaled], train_fit['misstate'])
    prediction = model.predict_proba(valid[raw_scaled])[:, 1]
    return roc_auc_score(valid['misstate'], prediction)

study_B_spw = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_B_spw.optimize(objective_spw_tuned, n_trials=50)
best_B_spw = study_B_spw.best_params
print("Καλύτερες ρυθμίσεις:", best_B_spw)

# Ξαναεκπαίδευση με τις καλύτερες ρυθμίσεις -> AUC και NDCG@1%
model_B_spw = XGBClassifier(random_state=RANDOM_STATE,
                            scale_pos_weight=best_B_spw['scale_pos_weight'],
                            n_estimators=best_B_spw['n_estimators'],
                            learning_rate=best_B_spw['learning_rate'],
                            max_depth=best_B_spw['max_depth'],
                            min_child_weight=best_B_spw['min_child_weight'],
                            subsample=best_B_spw['subsample'],
                            colsample_bytree=best_B_spw['colsample_bytree'])
model_B_spw.fit(train_fit[raw_scaled], train_fit['misstate'])
prediction_B_spw = model_B_spw.predict_proba(valid[raw_scaled])[:, 1]
auc_B_spw = roc_auc_score(valid['misstate'], prediction_B_spw)
ndcg_B_spw = ndcg_1pct(valid, prediction_B_spw)

print("scale_pos_weight=5, Optuna (Α):          AUC", round(auc_A_spw, 4), "| NDCG@1%", round(ndcg_A_spw, 4))
print("scale_pos_weight μέσα στο Optuna (Β):    AUC", round(auc_B_spw, 4), "| NDCG@1%", round(ndcg_B_spw, 4))

Καλύτερες ρυθμίσεις: {'scale_pos_weight': 1.2883333782360646, 'n_estimators': 1057, 'learning_rate': 0.011875020655065738, 'max_depth': 4, 'min_child_weight': 3.852260948765374, 'subsample': 0.6771006305632494, 'colsample_bytree': 0.8851320845005959}
scale_pos_weight=5, Optuna (Α):          AUC 0.7905 | NDCG@1% 0.1579
scale_pos_weight μέσα στο Optuna (Β):    AUC 0.7942 | NDCG@1% 0.142


### O συνδυασμός RUS 1:4 → SMOTE 1:1 με Optuna, εκδοχή Α

In [18]:
# Συνδυασμός RUS 1:4 -> SMOTE 1:1, εκδοχή Α: σταθερές αναλογίες, μέσος όρος 3 seeds σε κάθε δοκιμή
combo_seeds = [1, 21, 42]

def make_combo_train(seed, k_rus, k_smote):
    non_sample = non_frauds.sample(n=len(frauds) * k_rus, random_state=seed)
    train_c = pd.concat([frauds, non_sample])
    X_c = train_c[raw_scaled].fillna(medians)
    y_c = train_c['misstate']
    return SMOTE(sampling_strategy=1 / k_smote, random_state=seed).fit_resample(X_c, y_c)

# Τα 3 σύνολα εκπαίδευσης φτιάχνονται μία φορά (οι αναλογίες δεν αλλάζουν σε αυτή την εκδοχή)
combo_sets_A = []
for s in combo_seeds:
    combo_sets_A.append(make_combo_train(s, 4, 1))

def objective_combo_fixed(trial):
    n_estimators     = trial.suggest_int('n_estimators', 100, 2000)
    learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth        = trial.suggest_int('max_depth', 3, 12)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 7)
    subsample        = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)

    aucs = []
    for X_cs, y_cs in combo_sets_A:
        model = XGBClassifier(random_state=RANDOM_STATE,
                              n_estimators=n_estimators, learning_rate=learning_rate,
                              max_depth=max_depth, min_child_weight=min_child_weight,
                              subsample=subsample, colsample_bytree=colsample_bytree)
        model.fit(X_cs, y_cs)
        prediction = model.predict_proba(X_valid_filled)[:, 1]
        aucs.append(roc_auc_score(valid['misstate'], prediction))
    return np.mean(aucs)

study_A_combo = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_A_combo.optimize(objective_combo_fixed, n_trials=50)
best_A_combo = study_A_combo.best_params
print("Καλύτερες ρυθμίσεις:", best_A_combo)

# Σύγκριση στα ίδια 3 σύνολα: προεπιλογές έναντι καλύτερων ρυθμίσεων (μέσος όρος AUC και NDCG@1%)
default_scores = []
tuned_scores = []
for X_cs, y_cs in combo_sets_A:
    model_d = XGBClassifier(random_state=RANDOM_STATE)
    model_d.fit(X_cs, y_cs)
    pred_d = model_d.predict_proba(X_valid_filled)[:, 1]
    default_scores.append((roc_auc_score(valid['misstate'], pred_d), ndcg_1pct(valid, pred_d)))

    model_t = XGBClassifier(random_state=RANDOM_STATE,
                            n_estimators=best_A_combo['n_estimators'],
                            learning_rate=best_A_combo['learning_rate'],
                            max_depth=best_A_combo['max_depth'],
                            min_child_weight=best_A_combo['min_child_weight'],
                            subsample=best_A_combo['subsample'],
                            colsample_bytree=best_A_combo['colsample_bytree'])
    model_t.fit(X_cs, y_cs)
    pred_t = model_t.predict_proba(X_valid_filled)[:, 1]
    tuned_scores.append((roc_auc_score(valid['misstate'], pred_t), ndcg_1pct(valid, pred_t)))

default_scores = pd.DataFrame(default_scores, columns=['AUC', 'NDCG@1%'])
tuned_scores = pd.DataFrame(tuned_scores, columns=['AUC', 'NDCG@1%'])
print("Συνδυασμός, προεπιλογές (μ.ό. 3 seeds):  AUC", round(default_scores['AUC'].mean(), 4), "| NDCG@1%", round(default_scores['NDCG@1%'].mean(), 4))
print("Συνδυασμός, Optuna Α (μ.ό. 3 seeds):     AUC", round(tuned_scores['AUC'].mean(), 4), "| NDCG@1%", round(tuned_scores['NDCG@1%'].mean(), 4))

Καλύτερες ρυθμίσεις: {'n_estimators': 663, 'learning_rate': 0.010935840227283439, 'max_depth': 6, 'min_child_weight': 3.395111983968026, 'subsample': 0.5806841598062513, 'colsample_bytree': 0.4494123127321008}
Συνδυασμός, προεπιλογές (μ.ό. 3 seeds):  AUC 0.7419 | NDCG@1% 0.1028
Συνδυασμός, Optuna Α (μ.ό. 3 seeds):     AUC 0.7636 | NDCG@1% 0.126


In [ ]:
# Συνδυασμός, εκδοχή Β: το Optuna ψάχνει αναλογίες undersampling / SMOTE
def objective_combo_tuned(trial):
    k_rus   = trial.suggest_int('k_rus', 2, 10)             # μη απάτες ανά απάτη μετά το undersampling
    k_smote = trial.suggest_int('k_smote', 1, k_rus - 1)    # μετά το SMOTE: πάντα πιο ισορροπημένο
    n_estimators     = trial.suggest_int('n_estimators', 100, 2000)
    learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
    max_depth        = trial.suggest_int('max_depth', 3, 12)
    min_child_weight = trial.suggest_float('min_child_weight', 1, 7)
    subsample        = trial.suggest_float('subsample', 0.4, 1.0)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.4, 1.0)

    aucs = []
    for s in combo_seeds:
        X_cs, y_cs = make_combo_train(s, k_rus, k_smote)    # τα σύνολα φτιάχνονται εδώ, γιατί οι αναλογίες αλλάζουν
        model = XGBClassifier(random_state=RANDOM_STATE,
                              n_estimators=n_estimators, learning_rate=learning_rate,
                              max_depth=max_depth, min_child_weight=min_child_weight,
                              subsample=subsample, colsample_bytree=colsample_bytree)
        model.fit(X_cs, y_cs)
        prediction = model.predict_proba(X_valid_filled)[:, 1]
        aucs.append(roc_auc_score(valid['misstate'], prediction))
    return np.mean(aucs)

study_B_combo = optuna.create_study(direction='maximize', sampler=TPESampler(seed=RANDOM_STATE))
study_B_combo.optimize(objective_combo_tuned, n_trials=50)
best_B_combo = study_B_combo.best_params
print("Καλύτερες ρυθμίσεις:", best_B_combo)

# Ξαναεκπαίδευση με τις καλύτερες ρυθμίσεις στα 3 seeds -> μέσος όρος AUC και NDCG@1%
scores_B_combo = []
for s in combo_seeds:
    X_cs, y_cs = make_combo_train(s, best_B_combo['k_rus'], best_B_combo['k_smote'])
    model_t = XGBClassifier(random_state=RANDOM_STATE,
                            n_estimators=best_B_combo['n_estimators'],
                            learning_rate=best_B_combo['learning_rate'],
                            max_depth=best_B_combo['max_depth'],
                            min_child_weight=best_B_combo['min_child_weight'],
                            subsample=best_B_combo['subsample'],
                            colsample_bytree=best_B_combo['colsample_bytree'])
    model_t.fit(X_cs, y_cs)
    pred_t = model_t.predict_proba(X_valid_filled)[:, 1]
    scores_B_combo.append((roc_auc_score(valid['misstate'], pred_t), ndcg_1pct(valid, pred_t)))

scores_B_combo = pd.DataFrame(scores_B_combo, columns=['AUC', 'NDCG@1%'])
print("Συνδυασμός, Optuna Α (μ.ό. 3 seeds):  AUC", round(tuned_scores['AUC'].mean(), 4), "| NDCG@1%", round(tuned_scores['NDCG@1%'].mean(), 4))
print("Συνδυασμός, Optuna Β (μ.ό. 3 seeds):  AUC", round(scores_B_combo['AUC'].mean(), 4), "| NDCG@1%", round(scores_B_combo['NDCG@1%'].mean(), 4))

Καλύτερες ρυθμίσεις: {'k_rus': 7, 'k_smote': 6, 'n_estimators': 392, 'learning_rate': 0.010067297552614205, 'max_depth': 8, 'min_child_weight': 6.645365039844616, 'subsample': 0.6512542525009725, 'colsample_bytree': 0.6471732661102518}
Συνδυασμός, Optuna Α (μ.ό. 3 seeds):  AUC 0.7636 | NDCG@1% 0.126
Συνδυασμός, Optuna Β (μ.ό. 3 seeds):  AUC 0.7799 | NDCG@1% 0.2007
